Objetivo del proyecto: Queremos clasificar una nueva imagen segun su categoria dentro del dataset.

- Origen de los datos: Kaggle
- Tarea: Convertir las imagenes en datos para poder trabajar con ellas
- Atributo a predecir: clase
- Tipos de atributo: Numericos y

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd # Crear y manipular nuestra tabla de datos, el DataFrame.
import numpy as np # Trabajar con las matrices de píxeles y calcular las medias RGB.
from pathlib import Path # Localizar las ocho carpetas y recorrer los archivos de imágenes.
from PIL import Image #	Abrir las imágenes y obtener sus dimensiones y píxeles.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, balanced_accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.base import clone

from features_hls import extract_hls_features


RANDOM_STATE = 42

print("100% | Entorn preparat.")

100% | Entorn preparat.


In [ ]:
data_dir = Path("/home/fran/.cache/kagglehub/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/versions/1")

categorias = []
data = []

for carpeta in data_dir.iterdir():
    if carpeta.is_dir():
        categorias.append(carpeta.name)
        for carpeta_interior in carpeta.iterdir():
            
            imagenes = list(carpeta_interior.iterdir())
            # Nos devuelve algo como "Path("../data/polyps/polyps/imagen1.jpg"),

            # Path("../data/polyps/polyps/imagen2.jpg"),"
            print(carpeta_interior.name, ":", len(imagenes))
            
            for imagen in imagenes:
                img = Image.open(imagen)

                matriz = np.array(img.convert("RGB")) # (576, 720, 3) donde 3 es el RGB

                #print(matriz.shape)
                
                # Seleccionamos los valores rojos de todos los píxeles
                rojo = matriz[:, :, 0]
                verde = matriz[:, :, 1]
                azul = matriz[:, :, 2]

                # calculamos la media porque si fuera la suma nos daria el total de rojo pero no queremos eso
                rojo_medio = rojo.mean()
                verde_medio = verde.mean()
                azul_medio = azul.mean()

                #print("Rojo medio:", rojo_medio)
                #print("Verde medio:", verde_medio)
                #print("Azul medio:", azul_medio)

                rojo_std = rojo.std()
                verde_std = verde.std()
                azul_std = azul.std()

                data.append({
                    "image_path": imagen.relative_to(data_dir).as_posix(),
                    "clase": carpeta.name,
                    "rojo_medio": rojo_medio,
                    "verde_medio": verde_medio,
                    "azul_medio": azul_medio,
                    "rojo_std": rojo_std,
                    "verde_std": verde_std,
                    "azul_std": azul_std
                })

df = pd.DataFrame(data)

display(df.head())
df.describe()
#print(categorias)

data_hls = []

for ruta in df["image_path"]:
    imagen = data_dir / ruta
    caracteristicas = extract_hls_features(imagen)
    data_hls.append(caracteristicas)
    
df_hls = pd.DataFrame(data_hls)

df_hls["image_path"] = df["image_path"].to_numpy()
df_hls["clase"] = df["clase"].to_numpy()

print(df_hls.shape)
print(df_hls.isna().sum())
print(df_hls["clase"].value_counts())
print(df_hls["image_path"].duplicated().sum())

print(df["image_path"].equals(df_hls["image_path"]))

# Mirar de tener CSV con los datos de los RGB y HLS procesados
df.to_csv("rgb_features.csv", index=False)
df_hls.to_csv("hls_features.csv", index=False)
df = pd.read_csv("rgb_features.csv")
df_hls = pd.read_csv("hls_features.csv")

# Entrenar modelos como Random Forest, KNN o SVM.
#
# Para cada imagen nueva, obtener:
# % de dyed-lifted-polyps
# % de dyed-resection-margins
# % de esophagitis
# % de normal-cecum
# % de normal-pylorus
# % de normal-z-line
# % de polyps
# % de ulcerative-colitis

# Queremos utilizar las curvas de aprendizaje y validación
# para encontrar el mejor modelo y sus hiperparámetros,
# evitando el sobreajuste (overfitting) y el subajuste (underfitting).

# Tendremos que evaluar el modelo usando Accuracy, F1-macro y AUC multiclase.

# Accuracy mide el porcentaje de imágenes clasificadas correctamente.

# F1-macro combina Precision y Recall, dando la misma importancia a cada clase.

# AUC-ROC multiclase mide la capacidad del modelo para distinguir
# cada clase del resto a través de diferentes umbrales.


#########################
# DEEP LEARNING

# CNN para que aprenda de las diferentes imagenes y sepa clasificarlas
# YOLO DETECTION para detectar polipos

# La CNN se evaluará con Accuracy, F1-macro y AUC multiclase.
# YOLO Detection se evaluará con mAP, Precision y Recall.


# Finalmente comparar CNN con los modelos de machine learning

esophagitis : 1000
ulcerative-colitis : 1000
dyed-resection-margins : 1000
normal-z-line : 1000
normal-cecum : 1000
normal-pylorus : 1000
polyps : 1000
dyed-lifted-polyps : 1000


,image_path,clase,rojo_medio,verde_medio,azul_medio,rojo_std,verde_std,azul_std
0,esophagitis/esophagitis/08fe1ae5-193b-4df3-bd6...,esophagitis,143.553840,89.346442,85.676377,94.490250,59.500007,56.394077
1,esophagitis/esophagitis/e1d4308b-1fe9-4c50-ac3...,esophagitis,185.785549,143.955847,146.557935,71.775909,65.779551,66.609245
2,esophagitis/esophagitis/d6be9846-972d-4293-9c8...,esophagitis,136.000996,103.383585,98.504425,56.670453,55.069547,52.923576
3,esophagitis/esophagitis/14141a68-d121-478e-85e...,esophagitis,119.133834,77.264586,77.006903,80.661461,52.349438,52.597412
4,esophagitis/esophagitis/15f4c12b-b487-434c-bcf...,esophagitis,124.764456,75.390568,77.450348,82.807206,48.263455,50.064207


,h_sin_mean,h_cos_mean,h_concentration,l_mean,l_std,s_mean,s_std
0,0.053275,0.986009,0.987447,114.728912,74.723710,88.754272,67.963818
1,-0.045789,0.973705,0.974781,164.200353,68.026117,99.699786,54.663890
2,0.139093,0.927138,0.937514,116.338062,54.161027,64.127683,48.539218
3,0.013710,0.988296,0.988391,97.760330,65.872403,56.913027,43.340731
4,-0.020714,0.988114,0.988331,99.995991,65.201126,64.719862,50.627501


(8000, 7)


Modelos

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculamos las métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"Accuracy: {accuracy:.2%}")
    print(f"Precision: {precision:.2%}")
    print(f"Recall: {recall:.2%}")
    print(f"F1-score: {f1:.2%}")

    ConfusionMatrixDisplay.from_predictions(
        y_train,
        y_pred,
        display_labels=df["clase"].unique(),
        cmap='Blues'
    )

    #Matriz de 8x8
    plt.title("Matriz de confusión RGB")
    plt.tight_layout()
    plt.show()


# Features
X_rgb = df.drop(["clase", "image_path"], axis=1)
X_hls = df_hls.drop(["clase", "image_path"], axis=1)

# Target (Ground Truth)
y = df["clase"]

# Comprobamos que las imágenes y clases coinciden
assert df["image_path"].equals(df_hls["image_path"])
assert df["clase"].equals(df_hls["clase"])

# Mismas particiones para RGB y HLS
X_rgb_train, X_rgb_test, X_hls_train, X_hls_test, y_train, y_test = train_test_split(
    X_rgb,
    X_hls,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# =========================
# RGB
# =========================

print("Dummy Classifier - RGB:")
evaluate_model(
    DummyClassifier(strategy="most_frequent"),
    X_rgb_train, y_train, X_rgb_test, y_test
)

print("Random Forest - RGB:")
evaluate_model(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_rgb_train, y_train, X_rgb_test, y_test
)

# =========================
# HLS
# =========================

print("Dummy Classifier - HLS:")
evaluate_model(
    DummyClassifier(strategy="most_frequent"),
    X_hls_train, y_train, X_hls_test, y_test
)

print("Random Forest - HLS:")
evaluate_model(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_hls_train, y_train, X_hls_test, y_test
)

Dummy Classifier:
Accuracy: 12.50%
Precision: 1.56%
Recall: 12.50%
F1-score: 2.78%


ValueError: Found input variables with inconsistent numbers of samples: [6400, 1600, 6400]